# Building the Three-Layer Metrics Stack

`raw (event grain)` -> `clean` -> `video_life (video x country)` -> `channel`

Every query below is executed live with DuckDB. Outputs are real, not typed in.


In [1]:
import duckdb
import pandas as pd

con = duckdb.connect()
CSV = '../data/youtube.csv'


## Step 1 — Three-step cleaning (`sql/01-clean.sql`)

Isolate `#NAME?`, isolate the 11 IDs with inconsistent publish attributes,
dedup same-day duplicate scrapes.

In [2]:
con.sql(f"""
    CREATE OR REPLACE TABLE bad_pub AS
    SELECT video_id
    FROM read_csv_auto('{CSV}')
    WHERE video_id <> '#NAME?'
    GROUP BY 1
    HAVING COUNT(DISTINCT publish_date) > 1
        OR COUNT(DISTINCT time_frame) > 1
""")

con.sql(f"""
    CREATE OR REPLACE TABLE clean AS
    SELECT * EXCLUDE (rn, trend_dt), trend_dt FROM (
        SELECT *,
               strptime(trending_date, '%y.%d.%m')::DATE AS trend_dt,
               ROW_NUMBER() OVER (
                   PARTITION BY video_id, publish_country, strptime(trending_date, '%y.%d.%m')::DATE
                   ORDER BY views DESC, likes DESC
               ) AS rn
        FROM read_csv_auto('{CSV}')
        WHERE video_id <> '#NAME?'
          AND video_id NOT IN (SELECT video_id FROM bad_pub)
    ) WHERE rn = 1
""")

con.sql("SELECT COUNT(*) AS n_clean_rows FROM clean").df()

,n_clean_rows
0,159377


## Step 2 — Video-level table (`sql/02-video-life.sql`)

Aggregates to video x country grain. Attribute columns use `ARG_MAX(col,
trend_dt)` for determinism, not `ANY_VALUE`.

In [3]:
con.sql("""
    CREATE OR REPLACE TABLE video_life AS
    SELECT video_id, publish_country,
           ARG_MAX(channel_title, trend_dt)         AS channel_title,
           ARG_MAX(category_id, trend_dt)           AS category_id,
           ARG_MAX(published_day_of_week, trend_dt) AS pub_dow,
           ARG_MAX(time_frame, trend_dt)            AS pub_hour_utc,
           MIN(trend_dt)                             AS first_trend_day,
           MAX(trend_dt)                             AS last_trend_day,
           COUNT(DISTINCT trend_dt)                  AS days_on_trending,
           (MAX(trend_dt) - MIN(trend_dt)) + 1        AS trending_span,
           CASE WHEN COUNT(DISTINCT trend_dt) >= 2 THEN 1 ELSE 0 END AS survived_day1,
           MAX(views)                                AS peak_views,
           MAX(likes)                                AS peak_likes,
           MAX(comment_count)                        AS peak_comments
    FROM clean
    GROUP BY 1, 2
""")

con.sql("""
    CREATE OR REPLACE TABLE scrape_missing_days AS
    SELECT UNNEST(['2018-01-10','2018-01-11','2018-04-08','2018-04-09',
                   '2018-04-10','2018-04-11','2018-04-12','2018-04-13'])::DATE AS d
""")

con.sql("ALTER TABLE video_life ADD COLUMN IF NOT EXISTS gap_real INT")
con.sql("ALTER TABLE video_life ADD COLUMN IF NOT EXISTS gap_scrape INT")

con.sql("""
    UPDATE video_life v SET
        gap_real = CASE WHEN trending_span - days_on_trending
                            - (SELECT COUNT(*) FROM scrape_missing_days m
                               WHERE m.d BETWEEN v.first_trend_day AND v.last_trend_day) > 0
                        THEN 1 ELSE 0 END
""")

con.sql("""
    UPDATE video_life SET
        gap_scrape = CASE WHEN trending_span > days_on_trending AND gap_real = 0 THEN 1 ELSE 0 END
""")

con.sql("SELECT COUNT(*) AS n_video_life_rows FROM video_life").df()

,n_video_life_rows
0,63783


### Verifying real gap vs. scrape-attributable gap

1,262 multi-day videos have `trending_span > days_on_trending`. Of those,
413 have every missing day falling on the 8 known scraper-outage days
(`gap_scrape`); the remaining 849 have at least one unexplained missing
day (`gap_real`).

In [4]:
con.sql("""
    SELECT
        SUM(CASE WHEN trending_span > days_on_trending THEN 1 ELSE 0 END) AS any_gap,
        SUM(gap_real) AS gap_real_count,
        SUM(gap_scrape) AS gap_scrape_count
    FROM video_life
""").df()

,any_gap,gap_real_count,gap_scrape_count
0,1262.0,849.0,413.0


### Verifying `ARG_MAX` determinism vs. `ANY_VALUE` drift

Testing two scenarios to isolate exactly where the drift comes from:
(1) querying the same, already-materialized `clean` table repeatedly,
vs. (2) rebuilding `clean` from the raw CSV from scratch each time, then
querying.

In [5]:
print('--- Scenario 1: same materialized clean table, repeated ANY_VALUE queries ---')
for run in range(1, 4):
    r = con.sql("""
        SELECT COUNT(DISTINCT channel_title) FROM (
            SELECT video_id, publish_country, ANY_VALUE(channel_title) AS channel_title
            FROM clean GROUP BY 1, 2
        )
    """).fetchone()[0]
    print(f'Run {run}: {r}')

--- Scenario 1: same materialized clean table, repeated ANY_VALUE queries ---
Run 1: 12252
Run 2: 12252
Run 3: 12252


In [6]:
print('--- Scenario 2: rebuilding clean from CSV each time, then ANY_VALUE ---')
for run in range(1, 4):
    con.sql(f"""
        CREATE OR REPLACE TABLE clean_rebuild AS
        SELECT * EXCLUDE (rn, trend_dt), trend_dt FROM (
            SELECT *,
                   strptime(trending_date, '%y.%d.%m')::DATE AS trend_dt,
                   ROW_NUMBER() OVER (
                       PARTITION BY video_id, publish_country, strptime(trending_date, '%y.%d.%m')::DATE
                       ORDER BY views DESC, likes DESC
                   ) AS rn
            FROM read_csv_auto('{CSV}')
            WHERE video_id <> '#NAME?'
              AND video_id NOT IN (SELECT video_id FROM bad_pub)
        ) WHERE rn = 1
    """)
    r = con.sql("""
        SELECT COUNT(DISTINCT channel_title) FROM (
            SELECT video_id, publish_country, ANY_VALUE(channel_title) AS channel_title
            FROM clean_rebuild GROUP BY 1, 2
        )
    """).fetchone()[0]
    print(f'Run {run}: {r}')

--- Scenario 2: rebuilding clean from CSV each time, then ANY_VALUE ---


Run 1: 12254


Run 2: 12256


Run 3: 12255


In [7]:
con.sql("""
    SELECT COUNT(DISTINCT channel_title) FROM (
        SELECT video_id, publish_country, ARG_MAX(channel_title, trend_dt) AS channel_title
        FROM clean GROUP BY 1, 2
    )
""").df()

,count(DISTINCT channel_title)
0,12253


**Conclusion**: querying a fixed, already-built table repeatedly gives a
stable (if arbitrary) `ANY_VALUE` result, because physical row order
doesn't change between queries. The drift shows up specifically when the
source table is rebuilt from the CSV each time — DuckDB's parallel CSV
reader can finish chunks in a slightly different order on each build,
changing which row `ANY_VALUE` happens to land on. This matters because a
CI pipeline rebuilding these tables from scratch on every run is exactly
the "rebuild from CSV" scenario. `ARG_MAX(channel_title, trend_dt)` is
stable in both scenarios because it always resolves to the row with the
latest `trend_dt`, regardless of physical row order.

## Step 3 — Channel-level table (`sql/03-channel.sql`)

In [8]:
con.sql("""
    CREATE OR REPLACE TABLE channel AS
    SELECT channel_title,
           COUNT(DISTINCT video_id)        AS trending_videos,
           AVG(days_on_trending)           AS avg_video_life,
           AVG(survived_day1)              AS multi_day_rate,
           COUNT(DISTINCT publish_country) AS countries_reached
    FROM video_life
    GROUP BY 1
""")

con.sql("SELECT COUNT(*) AS n_channel_rows FROM channel").df()

,n_channel_rows
0,12253


### The channel-frequency reversal

Channels that trend most often actually have the *shortest* average
per-video lifespan — they rely on publish volume, not per-video staying
power. All top channels below are daily-upload institutional accounts.

In [9]:
con.sql("""
    SELECT channel_title, trending_videos, avg_video_life
    FROM channel
    ORDER BY trending_videos DESC
    LIMIT 5
""").df()

,channel_title,trending_videos,avg_video_life
0,VikatanTV,206,1.091483
1,The Late Show with Stephen Colbert,203,2.184300
2,Elhiwar Ettounsi,196,1.178862
3,ESPN,183,2.186916
4,CNN,166,2.620321


## Step 4 — Daily view velocity (`sql/04-velocity.sql`)

Window function `LAG`, grain stays event-level (video x country x day) —
this adds a column, it does not aggregate rows.

In [10]:
con.sql("""
    SELECT video_id, publish_country, trend_dt, views,
           views - LAG(views) OVER (
               PARTITION BY video_id, publish_country ORDER BY trend_dt
           ) AS views_gained_today
    FROM clean
    WHERE video_id = 'NooW_RbfdWI' AND publish_country = 'GB'
    ORDER BY trend_dt
    LIMIT 6
""").df()

,video_id,publish_country,trend_dt,views,views_gained_today
0,NooW_RbfdWI,GB,2018-02-05,1999326,<NA>
1,NooW_RbfdWI,GB,2018-02-06,8293323,6293997
2,NooW_RbfdWI,GB,2018-02-07,11883172,3589849
3,NooW_RbfdWI,GB,2018-02-08,14297585,2414413
4,NooW_RbfdWI,GB,2018-02-09,16344681,2047096
5,NooW_RbfdWI,GB,2018-02-10,18087776,1743095


**Conclusion**: this is the decay curve — a large initial jump, then a
steadily shrinking daily increment as the video ages off the trending list.

## Summary

| Table | Grain | Rows |
|---|---|---|
| `clean` | video x country x day (event grain) | 159,377 |
| `video_life` | video x country | 63,783 |
| `channel` | channel | 12,253 |

All three CSVs exported to `data/exports/` via `src/build_tables.py`.
